In [1]:
!pip install -q langchain langchain-groq langchain-community youtube-transcript-api chromadb sentence-transformers tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

In [3]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [4]:
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_2997/2549306482.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_2997/2549306482.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [8]:
from youtube_transcript_api import YouTubeTranscriptApi   # currently efficient with youtube videos with english speaker
import re

def extract_video_id(url: str) -> str:
    match = re.search(r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})", url)
    if not match:
        raise ValueError("Could not extract video ID from URL")
    return match.group(1)

def get_transcript(video_id: str):
    try:
        ytt_api = YouTubeTranscriptApi()
        fetched = ytt_api.fetch(video_id)
        # Convert to plain list of dicts, same shape as before
        return fetched.to_raw_data()  # list of {text, start, duration}
    except Exception as e:
        raise RuntimeError(f"No transcript available for this video: {e}")

In [10]:
url = "https://www.youtube.com/watch?v=g-TsjbaAXes"
video_id = extract_video_id(url)
transcript = get_transcript(video_id)

print(f"Fetched {len(transcript)} transcript segments")
print(transcript[:3])

Fetched 671 transcript segments
[{'text': 'Ladies and gentlemen,', 'start': 0.96, 'duration': 5.68}, {'text': 'I have made a lot of videos.', 'start': 3.679, 'duration': 5.92}, {'text': 'I have covered a lot of games.', 'start': 6.64, 'duration': 6.56}]


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def transcript_to_documents(transcript, video_id):
    """Combine transcript segments into chunks, keeping timestamp metadata"""
    full_text_with_time = [(seg["text"], seg["start"]) for seg in transcript]

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

    # Join all text for splitting
    combined_text = " ".join(text for text, _ in full_text_with_time)
    chunks = splitter.split_text(combined_text)

    # Approximate a timestamp for each chunk (first matching segment)
    documents = []
    for chunk in chunks:
        first_words = chunk[:30]
        matching_ts = next((ts for text, ts in full_text_with_time if first_words[:15] in text), 0)
        documents.append(Document(
            page_content=chunk,
            metadata={"video_id": video_id, "timestamp": matching_ts}
        ))
    return documents

docs = transcript_to_documents(transcript, video_id)
print(f"Created {len(docs)} chunks")
print(docs[0].page_content[:200])
print(docs[0].metadata)

Created 27 chunks
Ladies and gentlemen, I have made a lot of videos. I have covered a lot of games. I have made a lot of titles. Some they say are clickbaited or exaggerated. But today's is not far from it. And I will 
{'video_id': 'g-TsjbaAXes', 'timestamp': 0.96}


In [15]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,  # the HuggingFace embeddings object
    collection_name=f"video_{video_id}"
)

print(f"Stored {vectorstore._collection.count()} chunks in the vector store")

Stored 27 chunks in the vector store


In [17]:
# sanity check retrieval works
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
results = retriever.invoke("What is this video about?")

for r in results:
    print(f"[{r.metadata['timestamp']:.1f}s] {r.page_content[:150]}...\n")


[1.0s] Ladies and gentlemen, I have made a lot of videos. I have covered a lot of games. I have made a lot of titles. Some they say are clickbaited or exagge...

[0.0s] that continues the attack for white. And these computer games are so bananas, guys. You have been watching for 10 minutes. Some idiots left the video ...

[311.1s] you are going to watch this video? How many people have watched this video? When you've clicked on it, maybe a couple thousand. All right, maybe I fel...



In [18]:
from langchain_core.prompts import ChatPromptTemplate

map_prompt = ChatPromptTemplate.from_template(
    "Summarize the following transcript excerpt in 2-3 sentences, "
    "focusing on the key points:\n\n{text}"
)

def summarize_chunk(doc):
    chain = map_prompt | llm
    result = chain.invoke({"text": doc.page_content})
    return result.content

chunk_summaries = [summarize_chunk(doc) for doc in docs]  # contains all chunks summaries
print(f"Generated {len(chunk_summaries)} chunk summaries")
print(chunk_summaries[0])

Generated 27 chunk summaries
The speaker introduces a video showcasing a remarkable game of chess between two high-rated computer engines, Komodo Dragon and Scorpio, with ratings of 3650 and 3500, respectively. The game is played in a unique format called Fisher Random Chess 960 Freestyle, which starts with a randomized piece setup but quickly evolves into a standard structure. The speaker promises to walk the audience through the game, which they claim is "absolutely nuts" and features a thrilling engine battle.


In [19]:
# combining into final summary
reduce_prompt = ChatPromptTemplate.from_template(
    "You are given partial summaries of a YouTube video's transcript, in order.\n"
    "Combine them into one coherent, well-structured summary (5-8 sentences) "
    "covering the main points of the entire video.\n\n"
    "Partial summaries:\n{summaries}"
)

def generate_final_summary(chunk_summaries):
    combined = "\n\n".join(chunk_summaries)
    chain = reduce_prompt | llm
    result = chain.invoke({"summaries": combined})
    return result.content

final_summary = generate_final_summary(chunk_summaries)
print(final_summary)

The video showcases a remarkable game of chess between two high-rated computer engines, Komodo Dragon and Scorpio, played in the unique format of Fisher Random Chess 960 Freestyle. The game begins with the English opening and develops into a complex position, with both engines making strategic moves to gain an advantage. As the game progresses, the position becomes increasingly intricate, with the queens playing a key role in the battle. The engines make a series of sharp moves, including sacrifices and attacks, which ultimately lead to a dramatic and unexpected conclusion. The game reaches its climax as one of the engines promotes a third queen, setting up a winning endgame with three queens against a queen, bishop, and rook. The video concludes with the game ending in a surprising checkmate on the 72nd move, with the commentator inviting viewers to learn from the game and improve their own chess skills. Throughout the video, the commentator provides in-depth analysis and insights int

In [20]:
qa_prompt = ChatPromptTemplate.from_template(
    "You are answering questions about a YouTube video using only the transcript "
    "excerpts provided below. If the answer isn't in the excerpts, say so honestly "
    "rather than guessing.\n\n"
    "Transcript excerpts:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer clearly and concisely, in your own words."
)

def ask_question(question, k=4):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    relevant_docs = retriever.invoke(question)

    context = "\n\n".join(
        f"[{doc.metadata['timestamp']:.1f}s] {doc.page_content}"
        for doc in relevant_docs
    )

    chain = qa_prompt | llm
    result = chain.invoke({"context": context, "question": question})

    sources = [doc.metadata["timestamp"] for doc in relevant_docs]
    return result.content, sources

In [25]:
question = "What is the main point the speaker makes?"  # ask something relevant to your video
answer, sources = ask_question(question)

print("Answer:", answer)
print("\nSources (timestamps):", [f"{s:.1f}s" for s in sources])

Answer: The speaker is analyzing a chess game, highlighting the power of the queen and discussing strategies, but the main point is not explicitly stated in the provided excerpts. The speaker seems to be focusing on the game's progression and sharing their thoughts on various moves, rather than making a single, overarching point.

Sources (timestamps): ['1403.0s', '258.0s', '0.0s', '311.1s']
